# LeetCode 3569 — Maximize Count of Distinct Primes After Split bằng Python

- Link: https://leetcode.com/problems/maximize-count-of-distinct-primes-after-split/
- Độ khó: **Hard**
- Chủ đề: **Sàng nguyên tố**, **đoạn [lần đầu, lần cuối]**, **cây phân đoạn cộng đoạn – max toàn cục** (lazy), **heap xóa lười**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maximumCount(self, nums: List[int], queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Dùng **sàng Eratosthenes** để kiểm tra nguyên tố `O(1)` cho mọi giá trị ≤ `10^5`.
2. Biến đổi đại lượng "số nguyên tố khác nhau ở tiền tố + ở hậu tố" thành `D + (số số nguyên tố "bị cắt" bởi vị trí chia)`, trong đó `D` là số nguyên tố khác nhau của cả mảng.
3. Nhận ra mỗi số nguyên tố `p` ứng với một **đoạn** các vị trí chia `k ∈ [first_p + 1, last_p]` làm nó xuất hiện ở cả hai phía ⇒ bài toán **điểm bị phủ nhiều nhất**.
4. Duy trì các đoạn khi mảng thay đổi bằng **cây phân đoạn cộng đoạn + max toàn cục** và **heap xóa lười** để biết `first_p`, `last_p`.
5. Kiểm chứng bằng vét cạn và đo thời gian với `n = q = 5·10^4`.

## 2. Đề bài & ví dụ

Cho mảng `nums` độ dài `n` và các truy vấn `queries[i] = [idx, val]`. Với mỗi truy vấn:

1. Gán `nums[idx] = val` (thay đổi **giữ lại** cho các truy vấn sau).
2. Chọn `k` với `1 ≤ k < n` để chia mảng thành tiền tố `nums[0..k-1]` và hậu tố `nums[k..n-1]` (cả hai khác rỗng) sao cho **(số giá trị nguyên tố khác nhau trong tiền tố) + (số giá trị nguyên tố khác nhau trong hậu tố)** lớn nhất. Ghi lại giá trị lớn nhất đó.

**Ví dụ 1:** `nums = [2,1,3,1,2]`, `queries = [[1,2],[3,3]]` → `[3,4]`.

- Sau truy vấn 1: `[2,2,3,1,2]`, chia `[2] | [2,3,1,2]` → `1 + 2 = 3`.
- Sau truy vấn 2: `[2,2,3,3,2]`, chia `[2,2,3] | [3,2]` → `2 + 2 = 4`.

**Ví dụ 2:** `nums = [2,1,4]`, `queries = [[0,1]]` → `[0]` (không còn số nguyên tố).

**Ràng buộc:** `2 ≤ n ≤ 5·10^4`, `1 ≤ q ≤ 5·10^4`, `1 ≤ nums[i], val ≤ 10^5`.

In [1]:
from typing import List
import heapq
examples = [
    ([2, 1, 3, 1, 2], [[1, 2], [3, 3]], [3, 4]),
    ([2, 1, 4], [[0, 1]], [0]),
]
for nums, qs, exp in examples:
    print(nums, qs, '-> kỳ vọng', exp)

[2, 1, 3, 1, 2] [[1, 2], [3, 3]] -> kỳ vọng [3, 4]
[2, 1, 4] [[0, 1]] -> kỳ vọng [0]


## 3. Kiến thức nền

### 3.1. Sàng Eratosthenes

Đánh dấu mọi số từ 2 là "có thể nguyên tố"; với mỗi `i` còn được đánh dấu, gạch các bội `i·i, i·i + i, ...`. Dùng `bytearray` và gán lát cắt để nhanh trong Python. Nhớ: **1 không phải số nguyên tố**.

In [2]:
def sieve(N):
    s = bytearray([1]) * (N + 1)
    s[0] = s[1] = 0
    for i in range(2, int(N ** 0.5) + 1):
        if s[i]:
            s[i * i::i] = bytearray(len(range(i * i, N + 1, i)))
    return s

IS_PRIME = sieve(10**5)
print([x for x in range(1, 30) if IS_PRIME[x]], '| số nguyên tố <= 10^5:', sum(IS_PRIME))

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29] | số nguyên tố <= 10^5: 9592


### 3.2. Đếm khác nhau ở tiền tố và hậu tố

Với một mảng cố định, `pre[k]` = số nguyên tố khác nhau trong `nums[0..k-1]` tính dồn bằng một `set`; `suf[k]` tương tự từ phải. Đáp án cho mảng cố định = `max(pre[k] + suf[k])` với `1 ≤ k < n`. Tốn `O(n)`.

In [3]:
def best_split_linear(nums):
    n = len(nums)
    pre, seen = [0] * (n + 1), set()
    for i, x in enumerate(nums):
        if IS_PRIME[x]:
            seen.add(x)
        pre[i + 1] = len(seen)
    suf, seen = [0] * (n + 1), set()
    for i in range(n - 1, -1, -1):
        if IS_PRIME[nums[i]]:
            seen.add(nums[i])
        suf[i] = len(seen)
    return max(pre[k] + suf[k] for k in range(1, n))

print(best_split_linear([2, 2, 3, 1, 2]), best_split_linear([2, 2, 3, 3, 2]))

3 4


### 3.3. Heap xóa lười (lazy deletion)

Muốn biết vị trí **nhỏ nhất** và **lớn nhất** hiện tại của giá trị `p` khi các vị trí liên tục được thêm/bớt: giữ một min-heap và một max-heap (lưu `-i`) các vị trí. Khi bớt, **không xóa ngay**; lúc cần đọc đỉnh heap, nếu `nums[đỉnh] ≠ p` thì vị trí đó đã lỗi thời ⇒ `heappop` rồi xem tiếp.

In [4]:
arr = [5, 7, 5, 5]
mn = [0, 2, 3]; heapq.heapify(mn)
arr[0] = 9                       # vị trí 0 không còn là 5 (chưa xóa khỏi heap)
while arr[mn[0]] != 5:
    heapq.heappop(mn)            # dọn phần tử lỗi thời khi đọc
print('vị trí nhỏ nhất của 5 hiện tại:', mn[0])

vị trí nhỏ nhất của 5 hiện tại: 2


### 3.4. Cây phân đoạn: cộng một đoạn, hỏi max toàn mảng

Cây đầy đủ kích thước lũy thừa 2, mỗi nút lưu `t[x]` = max của đoạn nó quản lý (đã tính cả phần cộng `d[x]` áp lên cả nút). Cộng `v` cho `[l, r)`: đi từ lá lên như cây phân đoạn không đệ quy, cộng vào các nút phủ trọn; sau đó **cập nhật lại** các tổ tiên của hai biên: `t[p] = max(t[2p], t[2p+1]) + d[p]`. Max toàn cục là `t[1]`.

In [5]:
class RangeAddMax:
    def __init__(self, m):
        self.size = 1
        while self.size < max(1, m):
            self.size *= 2
        self.t = [0] * (2 * self.size)
        self.d = [0] * (2 * self.size)
        for i in range(m, self.size):
            self.t[self.size + i] = -10**9        # lá thừa không được chọn
        for i in range(self.size - 1, 0, -1):
            self.t[i] = max(self.t[2 * i], self.t[2 * i + 1])

    def add(self, l, r, v):                       # cộng v cho [l, r)
        if l >= r:
            return
        t, d = self.t, self.d
        l += self.size; r += self.size
        l0, r0 = l, r - 1
        while l < r:
            if l & 1:
                t[l] += v; d[l] += v; l += 1
            if r & 1:
                r -= 1; t[r] += v; d[r] += v
            l >>= 1; r >>= 1
        for p in (l0 >> 1, r0 >> 1):
            while p:
                t[p] = max(t[2 * p], t[2 * p + 1]) + d[p]
                p >>= 1

    def max_all(self):
        return self.t[1]

st = RangeAddMax(5)
st.add(0, 3, 1); st.add(2, 5, 1); st.add(2, 3, 5)
print('max toàn cục =', st.max_all())     # vị trí 2 nhận 1 + 1 + 5 = 7

max toàn cục = 7


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Gọi `D` là số nguyên tố khác nhau trong cả mảng. Với vị trí chia `k`, mỗi số nguyên tố `p` có mặt:

- chỉ ở tiền tố, hoặc chỉ ở hậu tố ⇒ đóng góp **1**;
- ở **cả hai** phía ⇒ đóng góp **2**.

Vậy `pre[k] + suf[k] = D + #{p : p xuất hiện ở cả hai phía}`.

`p` xuất hiện ở cả hai phía ⇔ `first_p ≤ k − 1` và `last_p ≥ k` ⇔ `k ∈ [first_p + 1, last_p]`. Mỗi `p` có ít nhất 2 lần xuất hiện tạo một **đoạn** các `k`; đáp án = `D + (số đoạn phủ một điểm k ∈ [1, n−1] nhiều nhất)`.

Tính tay sau truy vấn 2 ở ví dụ 1: `[2,2,3,3,2]`, `D = 2`.

| p | first | last | đoạn k |
|---|---|---|---|
| 2 | 0 | 4 | [1, 4] |
| 3 | 2 | 3 | [3, 3] |

Điểm `k = 3` bị phủ 2 lần ⇒ đáp án `2 + 2 = 4`.

In [6]:
def via_segments(nums):
    first, last = {}, {}
    for i, x in enumerate(nums):
        if IS_PRIME[x]:
            first.setdefault(x, i)
            last[x] = i
    n = len(nums)
    cover = [0] * (n + 1)
    for p in first:
        for k in range(first[p] + 1, last[p] + 1):
            cover[k] += 1
    return len(first) + max(cover[1:n]), {p: (first[p] + 1, last[p]) for p in first}

print(via_segments([2, 2, 3, 3, 2]))

(4, {2: (1, 4), 3: (3, 3)})


## 5. Cách 1 — Brute Force: thử mọi `k`, đếm bằng `set`

Mỗi truy vấn: cập nhật, rồi với mỗi `k` dựng hai tập nguyên tố của tiền tố và hậu tố.

In [7]:
def max_count_brute(nums, queries):
    nums = nums[:]
    res = []
    for idx, val in queries:
        nums[idx] = val
        n = len(nums)
        best = 0
        for k in range(1, n):
            left = {x for x in nums[:k] if IS_PRIME[x]}
            right = {x for x in nums[k:] if IS_PRIME[x]}
            best = max(best, len(left) + len(right))
        res.append(best)
    return res

for nums, qs, exp in examples:
    print(max_count_brute(nums, qs), '| kỳ vọng', exp)

[3, 4] | kỳ vọng [3, 4]
[0] | kỳ vọng [0]


**Độ phức tạp:** `O(q · n^2)` — với `n = q = 5·10^4` là `10^14`.

**Nút thắt:** (1) mỗi `k` dựng lại tập từ đầu; (2) mỗi truy vấn tính lại toàn mảng dù chỉ đổi một phần tử.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** `answer = D + max_k cover(k)` với `cover(k)` = số số nguyên tố có đoạn `[first_p + 1, last_p]` chứa `k`.

**Insight 2:** một truy vấn đổi `nums[idx]` từ `old` sang `val` chỉ ảnh hưởng **đoạn của `old`** và **đoạn của `val`**. Ta:

1. Gỡ đoạn cũ của `old` (cộng `−1`), bỏ `idx` khỏi tập vị trí của `old`, nếu `old` còn xuất hiện thì gắn đoạn mới (cộng `+1`).
2. Với `val`: gỡ đoạn cũ (nếu có), thêm `idx`, gắn đoạn mới.
3. Cập nhật `D` (giá trị nguyên tố xuất hiện lần đầu / biến mất).

**Insight 3:** cần `max_k cover(k)` sau mỗi cập nhật ⇒ cây phân đoạn **cộng đoạn – max toàn cục** (`O(log n)` mỗi thao tác). `first_p`/`last_p` lấy từ heap xóa lười.

Quy tắc "nếu… thì…":

- **Nếu** `old == val` **thì** không đổi gì.
- **Nếu** `p` chỉ xuất hiện 1 lần **thì** đoạn `[first+1, last]` rỗng (không cộng gì) nhưng `p` vẫn tính vào `D`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Mỗi truy vấn tính lại bằng `pre`/`suf` (O(n) mỗi truy vấn)

In [8]:
def max_count_linear(nums, queries):
    nums = nums[:]
    res = []
    for idx, val in queries:
        nums[idx] = val
        res.append(best_split_linear(nums))
    return res

for nums, qs, exp in examples:
    print(max_count_linear(nums, qs), '| kỳ vọng', exp)

[3, 4] | kỳ vọng [3, 4]
[0] | kỳ vọng [0]


**Độ phức tạp:** `O(q · n)` = `2.5·10^9` ở giới hạn — vẫn quá chậm, nhưng là oracle tốt cho dữ liệu vừa.

### 7.2. Cách 3 — Mỗi truy vấn: đoạn + mảng hiệu (O(n) mỗi truy vấn, cài theo góc nhìn mới)

Thay vì đếm tập, dùng trực tiếp công thức `D + max cover`: tìm `first`/`last` của mỗi số nguyên tố, cộng `+1` vào `diff[first+1]`, `−1` vào `diff[last+1]`, quét tiền tố. Vẫn `O(n)` mỗi truy vấn nhưng chuẩn bị cho cách 4.

In [9]:
def max_count_diff(nums, queries):
    nums = nums[:]
    n = len(nums)
    res = []
    for idx, val in queries:
        nums[idx] = val
        first, last = {}, {}
        for i, x in enumerate(nums):
            if IS_PRIME[x]:
                if x not in first:
                    first[x] = i
                last[x] = i
        diff = [0] * (n + 1)
        for p in first:
            diff[first[p] + 1] += 1
            diff[last[p] + 1] -= 1
        best = cur = 0
        for k in range(1, n):
            cur += diff[k]
            best = max(best, cur)
        res.append(len(first) + best)
    return res

for nums, qs, exp in examples:
    print(max_count_diff(nums, qs), '| kỳ vọng', exp)

[3, 4] | kỳ vọng [3, 4]
[0] | kỳ vọng [0]


Chú ý: vì `diff[0]` luôn bằng 0 (`first + 1 ≥ 1`), bắt đầu `cur` từ `k = 1` là đủ.

### 7.3. Cách 4 — Cây phân đoạn lazy + heap xóa lười (lời giải chính)

Chỉ số trên cây: vị trí chia `k ∈ [1, n−1]` ứng với lá `k − 1`; đoạn `k ∈ [f + 1, l]` ứng với lá `[f, l)` (nửa mở).

In [10]:
def max_count_segtree(nums, queries):
    nums = nums[:]
    n = len(nums)
    seg = RangeAddMax(n - 1)
    mn, mx, cnt = {}, {}, {}

    def ends(p):
        h = mn[p]
        while nums[h[0]] != p:
            heapq.heappop(h)
        g = mx[p]
        while nums[-g[0]] != p:
            heapq.heappop(g)
        return h[0], -g[0]

    def apply(p, v):
        f, l = ends(p)
        seg.add(f, l, v)            # k ∈ [f+1, l] <-> lá [f, l)

    for i, x in enumerate(nums):
        if IS_PRIME[x]:
            mn.setdefault(x, []).append(i)
            mx.setdefault(x, []).append(-i)
            cnt[x] = cnt.get(x, 0) + 1
    for p in mn:
        heapq.heapify(mn[p]); heapq.heapify(mx[p])
        apply(p, 1)

    res = []
    for idx, val in queries:
        old = nums[idx]
        if old != val:
            if IS_PRIME[old]:
                apply(old, -1)
                cnt[old] -= 1
                nums[idx] = -1                   # đánh dấu idx không còn là old
                if cnt[old] == 0:
                    del cnt[old], mn[old], mx[old]
                else:
                    apply(old, 1)
            nums[idx] = -1
            if IS_PRIME[val]:
                if val in cnt:
                    apply(val, -1)
                    cnt[val] += 1
                else:
                    cnt[val], mn[val], mx[val] = 1, [], []
            nums[idx] = val
            if IS_PRIME[val]:
                heapq.heappush(mn[val], idx)
                heapq.heappush(mx[val], -idx)
                apply(val, 1)
        res.append(len(cnt) + seg.max_all())
    return res

for nums, qs, exp in examples:
    print(max_count_segtree(nums, qs), '| kỳ vọng', exp)

[3, 4] | kỳ vọng [3, 4]
[0] | kỳ vọng [0]


Thứ tự rất quan trọng: khi gỡ đoạn cũ của `val`, vị trí `idx` **chưa** được tính là `val` (ta đặt `nums[idx] = -1` tạm thời), nên `ends(val)` trả về đúng đoạn đã cộng trước đó.

**Độ phức tạp:** khởi tạo `O(n log n)`; mỗi truy vấn tối đa 4 lần `add` `O(log n)` + thao tác heap `O(log n)` khấu hao ⇒ `O((n + q) log n)`.

## 8. TRACE từng bước

Trace cách 4 trên ví dụ 1: in đoạn của từng số nguyên tố, các thao tác `add`, `D`, và giá trị cover lớn nhất sau mỗi truy vấn.

In [11]:
def trace(nums, queries):
    nums = nums[:]
    n = len(nums)
    def segments():
        first, last = {}, {}
        for i, x in enumerate(nums):
            if IS_PRIME[x]:
                first.setdefault(x, i); last[x] = i
        return {p: (first[p] + 1, last[p]) for p in first}
    def cover_row(segs):
        row = []
        for k in range(1, n):
            row.append(sum(1 for a, b in segs.values() if a <= k <= b))
        return row
    segs = segments()
    print('ban đầu', nums, '| đoạn k:', segs, '| cover k=1..n-1:', cover_row(segs))
    for idx, val in queries:
        old = nums[idx]
        nums[idx] = val
        new = segments()
        print(f"\nTruy vấn [{idx}, {val}]: {old} -> {val}, mảng {nums}")
        for p in sorted(set(segs) | set(new)):
            if segs.get(p) != new.get(p):
                print(f"   số {p}: đoạn {segs.get(p)} -> {new.get(p)}  (gỡ cũ, gắn mới)")
        row = cover_row(new)
        print(f"   D = {len(new)}, cover k=1..{n-1} = {row}, max = {max(row)} -> đáp án {len(new) + max(row)}")
        segs = new

trace([2, 1, 3, 1, 2], [[1, 2], [3, 3]])

ban đầu [2, 1, 3, 1, 2] | đoạn k: {2: (1, 4), 3: (3, 2)} | cover k=1..n-1: [1, 1, 1, 1]

Truy vấn [1, 2]: 1 -> 2, mảng [2, 2, 3, 1, 2]
   D = 2, cover k=1..4 = [1, 1, 1, 1], max = 1 -> đáp án 3

Truy vấn [3, 3]: 1 -> 3, mảng [2, 2, 3, 3, 2]
   số 3: đoạn (3, 2) -> (3, 3)  (gỡ cũ, gắn mới)
   D = 2, cover k=1..4 = [1, 1, 2, 1], max = 2 -> đáp án 4


Ghi chú: đoạn dạng `(3, 2)` (đầu > cuối) là **rỗng** — số chỉ xuất hiện một lần, không bị cắt ở vị trí nào nhưng vẫn tính vào `D`.

In [12]:
# Đối chiếu cây phân đoạn: in t[1] sau từng thao tác add trong một truy vấn nhỏ
seg = RangeAddMax(4)
for (l, r, v) in [(0, 4, 1), (2, 3, 1), (0, 4, -1), (0, 2, 1)]:
    seg.add(l, r, v)
    leaves = []
    for i in range(4):          # giá trị thật ở lá = t[lá] + tổng d của các tổ tiên
        x, val = (seg.size + i) >> 1, seg.t[seg.size + i]
        while x:
            val += seg.d[x]; x >>= 1
        leaves.append(val)
    print(f"add({l},{r},{v:+}) -> lá = {leaves}, max toàn cục = {seg.max_all()}")

add(0,4,+1) -> lá = [1, 1, 1, 1], max toàn cục = 1
add(2,3,+1) -> lá = [1, 1, 2, 1], max toàn cục = 2
add(0,4,-1) -> lá = [0, 0, 1, 0], max toàn cục = 1
add(0,2,+1) -> lá = [1, 1, 1, 0], max toàn cục = 1


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def maximumCount(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        N = 10**5
        isp = bytearray([1]) * (N + 1)
        isp[0] = isp[1] = 0
        for i in range(2, int(N ** 0.5) + 1):
            if isp[i]:
                isp[i * i::i] = bytearray(len(range(i * i, N + 1, i)))

        nums = nums[:]
        n = len(nums)
        size = 1
        while size < n - 1:
            size *= 2
        t = [0] * (2 * size)
        d = [0] * (2 * size)
        for i in range(n - 1, size):
            t[size + i] = -10**9
        for i in range(size - 1, 0, -1):
            t[i] = max(t[2 * i], t[2 * i + 1])

        def add(l, r, v):
            if l >= r:
                return
            l += size; r += size
            l0, r0 = l, r - 1
            while l < r:
                if l & 1:
                    t[l] += v; d[l] += v; l += 1
                if r & 1:
                    r -= 1; t[r] += v; d[r] += v
                l >>= 1; r >>= 1
            for p in (l0 >> 1, r0 >> 1):
                while p:
                    t[p] = max(t[2 * p], t[2 * p + 1]) + d[p]
                    p >>= 1

        mn, mx, cnt = {}, {}, {}

        def apply(p, v):
            h = mn[p]
            while nums[h[0]] != p:
                heapq.heappop(h)
            g = mx[p]
            while nums[-g[0]] != p:
                heapq.heappop(g)
            add(h[0], -g[0], v)

        for i, x in enumerate(nums):
            if isp[x]:
                if x not in cnt:
                    mn[x], mx[x], cnt[x] = [], [], 0
                mn[x].append(i); mx[x].append(-i); cnt[x] += 1
        for p in cnt:
            heapq.heapify(mn[p]); heapq.heapify(mx[p])
            apply(p, 1)

        res = []
        for idx, val in queries:
            old = nums[idx]
            if old != val:
                if isp[old]:
                    apply(old, -1)
                    cnt[old] -= 1
                    nums[idx] = -1
                    if cnt[old] == 0:
                        del cnt[old], mn[old], mx[old]
                    else:
                        apply(old, 1)
                nums[idx] = -1
                if isp[val]:
                    if val in cnt:
                        apply(val, -1)
                        cnt[val] += 1
                    else:
                        cnt[val], mn[val], mx[val] = 1, [], []
                nums[idx] = val
                if isp[val]:
                    heapq.heappush(mn[val], idx)
                    heapq.heappush(mx[val], -idx)
                    apply(val, 1)
            res.append(len(cnt) + t[1])
        return res

for nums, qs, exp in examples:
    got = Solution().maximumCount(nums, qs)
    print(got, 'OK' if got == exp else 'SAI')

[3, 4] OK
[0] OK


## 10. Vì sao thuật toán đúng

1. **Công thức `D + cover`:** với mọi `k`, mỗi số nguyên tố có mặt trong mảng nằm ở ít nhất một phía. Nó được đếm hai lần ⇔ có mặt ở cả hai phía ⇔ `first ≤ k − 1 < k ≤ last` ⇔ `k ∈ [first + 1, last]`. Cộng lại: `pre[k] + suf[k] = D + cover(k)`.
2. **Bất biến cây phân đoạn:** sau mỗi truy vấn, giá trị tại lá `k − 1` bằng `cover(k)` với các đoạn hiện tại — vì ta luôn gỡ chính xác đoạn đã gắn (tính bằng `first`/`last` **trước** khi thay đổi) rồi gắn đoạn mới.
3. **Heap xóa lười đúng:** mọi vị trí còn mang giá trị `p` đều có trong heap của `p`; vị trí lỗi thời (giá trị khác `p`) bị bỏ qua khi lên đỉnh. Việc tạm đặt `nums[idx] = -1` đảm bảo `idx` không bị tính sai ở thời điểm gỡ đoạn cũ.
4. **`D = len(cnt)`:** `cnt` chỉ chứa số nguyên tố đang có mặt (xóa khi đếm về 0).
5. **Lá thừa = −∞** nên `t[1]` là max trên đúng `k ∈ [1, n − 1]`; nếu không có đoạn nào thì mọi lá thật bằng 0 ⇒ đáp án `D`.

## 11. Phân tích Time / Space Complexity

Gọi `n` độ dài mảng, `q` số truy vấn, `V = 10^5`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi k bằng set | `O(q · n^2)` | `O(n)` |
| 2. pre/suf mỗi truy vấn | `O(q · n)` | `O(n)` |
| 3. Đoạn + mảng hiệu mỗi truy vấn | `O(q · n)` | `O(n)` |
| 4. Cây phân đoạn + heap | `O(V log log V + (n + q) log n)` | `O(V + n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [14]:
impls = {
    'linear': max_count_linear,
    'diff': max_count_diff,
    'segtree': max_count_segtree,
    'submit': lambda a, q: Solution().maximumCount(a, q),
}
edge = [
    ([2, 2], [[0, 2]], [2]),                        # gán cùng giá trị; [2] | [2] -> 1 + 1
    ([1, 1], [[1, 7]], [1]),
    ([4, 6, 8], [[1, 9]], [0]),                     # không có số nguyên tố (9 không nguyên tố)
    ([2, 3, 5, 7], [[0, 7], [3, 2]], [4, 4]),       # [7,3,5,7]: D = 3, số 7 bị cắt -> 4; rồi [7,3,5,2]: D = 4
    ([99991, 99991, 1], [[2, 99991], [0, 1], [1, 1]], [2, 2, 1]),   # số nguyên tố lớn
]
for nums, qs, exp in examples + edge:
    for name, f in impls.items():
        got = f(nums, qs)
        assert got == exp, (name, nums, qs, got, exp)
    assert max_count_brute(nums, qs) == exp
print('Ví dụ đề và ca biên: tất cả đúng. 99991 nguyên tố?', bool(IS_PRIME[99991]))

Ví dụ đề và ca biên: tất cả đúng. 99991 nguyên tố? True


### 12.2. Đối chiếu ngẫu nhiên (giá trị nhỏ để trùng lặp nhiều)

In [15]:
import random
random.seed(3569)
cnt = 0
for _ in range(800):
    n = random.randint(2, 9)
    nums = [random.randint(1, 8) for _ in range(n)]
    qs = [[random.randrange(n), random.randint(1, 8)] for _ in range(random.randint(1, 25))]
    exp = max_count_brute(nums, qs)
    for name, f in impls.items():
        assert f(nums, qs) == exp, (name, nums, qs)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test ngẫu nhiên.')

Khớp vét cạn trên 800 bộ test ngẫu nhiên.


### 12.3. Ca lớn (`n = q = 5·10^4`)

In [16]:
import time
random.seed(50)
n = q = 5 * 10**4
small_primes = [p for p in range(2, 200) if IS_PRIME[p]]
for name, gen in [('giá trị ngẫu nhiên', lambda: random.randint(1, 10**5)),
                  ('ít số nguyên tố nhỏ', lambda: random.choice(small_primes + [1, 4]))]:
    nums = [gen() for _ in range(n)]
    qs = [[random.randrange(n), gen()] for _ in range(q)]
    t = time.time(); a = Solution().maximumCount(nums, qs); t1 = time.time() - t
    # kiểm tra vài truy vấn đầu bằng cách O(n)
    b = max_count_diff(nums, qs[:30])
    assert a[:30] == b
    print(f"{name}: {t1:.2f}s | 5 đáp án đầu {a[:5]}")

giá trị ngẫu nhiên: 0.17s | 5 đáp án đầu [4305, 4306, 4306, 4306, 4307]


ít số nguyên tố nhỏ: 1.70s | 5 đáp án đầu [92, 92, 92, 92, 92]


Khoảng 1 giây cho `5·10^4` truy vấn; cách `O(n)` mỗi truy vấn sẽ mất hàng chục phút ở cùng kích thước.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi 1 là số nguyên tố (sàng thiếu `s[1] = 0`)

In [17]:
bad_sieve = bytearray(IS_PRIME)
bad_sieve[1] = 1                          # SAI

def linear_with(sieve_arr, nums):
    n = len(nums)
    best = 0
    for k in range(1, n):
        best = max(best, len({x for x in nums[:k] if sieve_arr[x]}) + len({x for x in nums[k:] if sieve_arr[x]}))
    return best

print('sai :', linear_with(bad_sieve, [1, 1, 4]))   # 2
print('đúng:', linear_with(IS_PRIME, [1, 1, 4]))    # 0 (ví dụ 2)

sai : 2
đúng: 0


**Vì sao sai:** 1 không phải số nguyên tố; ví dụ 2 của đề kiểm tra đúng điều này.

### Lỗi 2 — Lệch biên đoạn: dùng `k ∈ [first, last]` thay vì `[first + 1, last]`

In [18]:
def wrong_range(nums):
    first, last = {}, {}
    for i, x in enumerate(nums):
        if IS_PRIME[x]:
            first.setdefault(x, i); last[x] = i
    n = len(nums)
    cover = [0] * (n + 1)
    for p in first:
        for k in range(first[p], last[p] + 1):      # SAI: k = first không cắt p
            cover[k] += 1
    return len(first) + max(cover[1:n])

random.seed(2)
found = None
for _ in range(2000):
    nums = [random.randint(1, 7) for _ in range(random.randint(2, 7))]
    if wrong_range(nums) != best_split_linear(nums):
        found = nums
        break
print('phản ví dụ:', found)
print('sai :', wrong_range(found))
print('đúng:', best_split_linear(found))

phản ví dụ: [7, 2, 6, 7]
sai : 4
đúng: 3


**Vì sao sai:** với `k = first`, tiền tố `nums[0..first−1]` **chưa** chứa `p` ⇒ `p` không bị cắt. Ở phản ví dụ trên, công thức sai làm một vị trí chia bị phủ thừa nên đáp án lớn hơn thực tế.

### Lỗi 3 — Gắn đoạn mới mà **quên gỡ đoạn cũ**

In [19]:
def wrong_no_remove(nums, queries):
    nums = nums[:]
    n = len(nums)
    seg = RangeAddMax(n - 1)
    def ends(p):
        pos = [i for i, x in enumerate(nums) if x == p]
        return pos[0], pos[-1]
    for p in {x for x in nums if IS_PRIME[x]}:
        f, l = ends(p); seg.add(f, l, 1)
    res = []
    for idx, val in queries:
        nums[idx] = val
        if IS_PRIME[val]:
            f, l = ends(val); seg.add(f, l, 1)       # SAI: không gỡ đoạn cũ của val/old
        res.append(len({x for x in nums if IS_PRIME[x]}) + seg.max_all())
    return res

print('sai :', wrong_no_remove([2, 1, 3, 1, 2], [[1, 2], [3, 3]]))
print('đúng:', Solution().maximumCount([2, 1, 3, 1, 2], [[1, 2], [3, 3]]))

sai : [4, 5]
đúng: [3, 4]


**Vì sao sai:** đoạn của mỗi số nguyên tố phải được đếm **đúng một lần**; quên gỡ khiến cùng một số được cộng nhiều lần.

### Lỗi 4 — Gỡ đoạn cũ của `val` **sau** khi đã gán `nums[idx] = val`

In [20]:
def remove_after_assign(nums, idx, val):
    """Mô phỏng ends(val) khi idx đã mang giá trị val: đoạn 'cũ' tính sai."""
    before = [i for i, x in enumerate(nums) if x == val]
    nums = nums[:]
    nums[idx] = val
    after = [i for i, x in enumerate(nums) if x == val]
    return (before[0], before[-1]), (after[0], after[-1])

print('đoạn đã gắn trước đó vs đoạn tính sau khi gán:', remove_after_assign([1, 1, 3, 3, 1], 0, 3))

đoạn đã gắn trước đó vs đoạn tính sau khi gán: ((2, 3), (0, 3))


**Vì sao sai:** đoạn đã gắn là `(2, 3)`; nếu tính "đoạn cũ" sau khi gán thì được `(0, 3)` và ta gỡ nhầm phạm vi `[0, 3)` ⇒ cây bị âm ở các lá `0, 1`. Lời giải chính gỡ trước, gán sau (dùng `nums[idx] = -1` tạm thời).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số phần tử khác nhau ở tiền tố + hậu tố" ⇒ `D + (số giá trị xuất hiện ở cả hai phía)`; mỗi giá trị ứng với đoạn `[first + 1, last]`.
- "Max số đoạn phủ một điểm" với cập nhật ⇒ **cây phân đoạn cộng đoạn – max**.
- "Vị trí nhỏ nhất/lớn nhất của giá trị khi có thêm/bớt" ⇒ `SortedList`, hoặc **heap xóa lười** (thư viện chuẩn).
- Cập nhật một phần tử ⇒ chỉ hai giá trị (`old`, `val`) bị ảnh hưởng: **gỡ cũ → sửa → gắn mới**.
- Kiểm tra nguyên tố nhiều lần ⇒ sàng một lần.

## 15. Biến thể của bài

1. Không có cập nhật ⇒ một lần quét `pre`/`suf` `O(n)`.
2. Đếm **mọi** giá trị khác nhau (không chỉ nguyên tố) ⇒ bỏ sàng, cùng khung.
3. Chia thành **ba** phần ⇒ đếm số lần mỗi giá trị bị "cắt" bởi hai vị trí; khó hơn đáng kể.
4. Bài liên quan: LC 2405, LC 1525 *Number of Good Ways to Split a String* (không cập nhật).

In [21]:
# Biến thể 2 (LC 1525 hướng tối đa hóa): mọi ký tự khác nhau, tối đa pre + suf
def best_split_chars(s):
    n = len(s)
    pre, seen = [0] * (n + 1), set()
    for i, ch in enumerate(s):
        seen.add(ch); pre[i + 1] = len(seen)
    suf, seen = [0] * (n + 1), set()
    for i in range(n - 1, -1, -1):
        seen.add(s[i]); suf[i] = len(seen)
    return max(pre[k] + suf[k] for k in range(1, n))

print(best_split_chars("aacaba"), best_split_chars("abcd"))

4 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Thử mọi k bằng set | `O(q n^2)` | `O(n)` | Oracle nhỏ |
| 2 | pre/suf mỗi truy vấn | `O(q n)` | `O(n)` | Oracle vừa |
| 3 | Đoạn + mảng hiệu | `O(q n)` | `O(n)` | Hiểu công thức `D + cover` |
| 4 | Cây phân đoạn + heap lười | `O((n+q) log n)` | `O(n + V)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `maximumCount([3, 3, 3], [[1, 2]])` = ?

**Bài 2 (tính tay):** `nums = [2, 3, 2, 3]`, không truy vấn cập nhật thực sự (`[[0, 2]]`) — đáp án?

**Bài 3 (điền TODO):** viết `segments(nums)` trả về dict `p -> (first+1, last)`.

**Bài 4 (tự viết lại):** viết `best_k(nums)` trả về **cả** giá trị lớn nhất và **một** vị trí `k` đạt được.

**Bài 5 (mini challenge):** biến thể 2 có cập nhật — đếm **mọi** giá trị khác nhau; sửa lời giải chính và kiểm chứng.

In [22]:
def segments_todo(nums):
    # TODO
    raise NotImplementedError

def best_k_todo(nums):
    # TODO
    raise NotImplementedError

def max_count_all_values_todo(nums, queries):
    # TODO: như Solution nhưng thay isp[x] bằng True
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Sau cập nhật: `[3, 2, 3]`. (2) `D = 2`; đoạn của 3: `k ∈ [1, 2]`. (3) Đáp án `2 + 1 = 3`.

**Bài 2.** (1) `D = 2`. (2) Đoạn 2: `[1, 2]`; đoạn 3: `[2, 3]`. (3) `k = 2` phủ 2 ⇒ đáp án 4.

**Bài 3.** (1) Duyệt một lần ghi `first` (lần đầu) và `last` (cập nhật liên tục). (2) Chỉ xét số nguyên tố. (3) Đoạn có thể rỗng nếu chỉ xuất hiện một lần.

**Bài 4.** (1) Dùng mảng hiệu. (2) Theo dõi `argmax`. (3) Kiểm tra `pre[k] + suf[k]` tại `k` đó.

**Bài 5.** (1) Bỏ điều kiện nguyên tố. (2) Phần còn lại giữ nguyên. (3) Đối chiếu với vét cạn trên mảng nhỏ.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
a1 = Solution().maximumCount([3, 3, 3], [[1, 2]])
a2 = Solution().maximumCount([2, 3, 2, 3], [[0, 2]])
assert a1 == [3] == max_count_brute([3, 3, 3], [[1, 2]])
assert a2 == [4] == max_count_brute([2, 3, 2, 3], [[0, 2]])
print('Bài 1:', a1, '| Bài 2:', a2)

Bài 1: [3] | Bài 2: [4]


In [24]:
# Bài 3, 4
def segments(nums):
    first, last = {}, {}
    for i, x in enumerate(nums):
        if IS_PRIME[x]:
            first.setdefault(x, i)
            last[x] = i
    return {p: (first[p] + 1, last[p]) for p in first}

def best_k(nums):
    segs = segments(nums)
    n = len(nums)
    diff = [0] * (n + 1)
    for a, b in segs.values():
        if a <= b:
            diff[a] += 1
            diff[b + 1] -= 1
    best, arg, cur = -1, None, 0
    for k in range(1, n):
        cur += diff[k]
        if cur > best:
            best, arg = cur, k
    return len(segs) + best, arg

random.seed(5)
for _ in range(300):
    nums = [random.randint(1, 10) for _ in range(random.randint(2, 10))]
    val, k = best_k(nums)
    left = {x for x in nums[:k] if IS_PRIME[x]}
    right = {x for x in nums[k:] if IS_PRIME[x]}
    assert val == len(left) + len(right) == best_split_linear(nums)
print('Bài 3:', segments([2, 3, 2, 3]), '| Bài 4:', best_k([2, 3, 2, 3]))

Bài 3: {2: (1, 2), 3: (2, 3)} | Bài 4: (4, 2)


In [25]:
# Bài 5: đếm mọi giá trị khác nhau, có cập nhật
def max_count_all_values(nums, queries):
    nums = nums[:]
    n = len(nums)
    seg = RangeAddMax(n - 1)
    mn, mx, cnt = {}, {}, {}
    def apply(p, v):
        h = mn[p]
        while nums[h[0]] != p:
            heapq.heappop(h)
        g = mx[p]
        while nums[-g[0]] != p:
            heapq.heappop(g)
        seg.add(h[0], -g[0], v)
    for i, x in enumerate(nums):
        mn.setdefault(x, []).append(i); mx.setdefault(x, []).append(-i); cnt[x] = cnt.get(x, 0) + 1
    for p in mn:
        heapq.heapify(mn[p]); heapq.heapify(mx[p]); apply(p, 1)
    res = []
    for idx, val in queries:
        old = nums[idx]
        if old != val:
            apply(old, -1); cnt[old] -= 1; nums[idx] = -1
            if cnt[old] == 0:
                del cnt[old], mn[old], mx[old]
            else:
                apply(old, 1)
            if val in cnt:
                apply(val, -1); cnt[val] += 1
            else:
                cnt[val], mn[val], mx[val] = 1, [], []
            nums[idx] = val
            heapq.heappush(mn[val], idx); heapq.heappush(mx[val], -idx); apply(val, 1)
        res.append(len(cnt) + seg.max_all())
    return res

def all_values_brute(nums, queries):
    nums = nums[:]
    out = []
    for idx, val in queries:
        nums[idx] = val
        out.append(max(len(set(nums[:k])) + len(set(nums[k:])) for k in range(1, len(nums))))
    return out

random.seed(6)
for _ in range(300):
    n = random.randint(2, 8)
    nums = [random.randint(1, 5) for _ in range(n)]
    qs = [[random.randrange(n), random.randint(1, 5)] for _ in range(10)]
    assert max_count_all_values(nums, qs) == all_values_brute(nums, qs)
print('Bài 5: khớp vét cạn; ví dụ', max_count_all_values([1, 2, 1, 3], [[3, 2], [0, 4]]))

Bài 5: khớp vét cạn; ví dụ [4, 4]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
answer = D + max_{1<=k<n} cover(k);  mỗi số nguyên tố p: cover += 1 trên k ∈ [first_p+1, last_p]
cây phân đoạn (lá k-1): add(first, last, ±1) nửa mở; max toàn cục = t[1]
first/last: min-heap & max-heap vị trí, xóa lười khi nums[top] != p
truy vấn (idx, val), old = nums[idx] != val:
    old nguyên tố: gỡ đoạn; cnt--; nums[idx] = -1; còn thì gắn lại
    val nguyên tố: gỡ đoạn cũ (nếu có) TRƯỚC khi gán; gán; push; gắn đoạn mới
```

**Checklist:**

- [ ] Sàng có `s[0] = s[1] = 0`?
- [ ] Đoạn `k ∈ [first + 1, last]` (không phải `[first, last]`)?
- [ ] Lá thừa của cây đặt −∞; chỉ xét `k ∈ [1, n−1]`?
- [ ] Luôn **gỡ đoạn cũ rồi mới gắn đoạn mới**, đúng thứ tự với việc gán `nums[idx]`?
- [ ] Bỏ qua truy vấn `old == val`?
- [ ] `D` cập nhật khi số nguyên tố xuất hiện lần đầu / biến mất?